In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: Sebuah Qiskit Function oleh ColibriTD
> **Note:** Qiskit Functions adalah fitur eksperimental yang tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini berstatus rilis pratinjau dan dapat berubah sewaktu-waktu.
## Ikhtisar
Solver Persamaan Diferensial Parsial (PDE) yang disajikan di sini merupakan bagian dari platform Quantum Innovative Computing Kit (QUICK) kami (QUICK-PDE), dan dikemas sebagai Qiskit Function. Dengan fungsi QUICK-PDE, kamu bisa menyelesaikan persamaan diferensial parsial khusus domain di IBM Quantum QPU. Fungsi ini didasarkan pada algoritma yang dijelaskan dalam [makalah deskripsi H-DES dari ColibriTD.](https://arxiv.org/abs/2410.01130) Algoritma ini mampu memecahkan masalah multi-fisika yang kompleks, dimulai dari Computational Fluid Dynamics (CFD) dan Material Deformation (MD), serta kasus penggunaan lainnya yang segera hadir.

Untuk menangani persamaan diferensial, solusi percobaan dikodekan sebagai kombinasi linier dari fungsi ortogonal (biasanya polinomial Chebyshev, lebih tepatnya $2^n$ di antaranya di mana $n$ adalah jumlah qubit yang mengkodekan fungsimu), diparametrikan oleh sudut-sudut dari Variable Quantum Circuit (VQC). Ansatz menghasilkan state yang mengkodekan fungsi, yang dievaluasi oleh observabel yang kombinasinya memungkinkan evaluasi fungsi di semua titik. Kamu kemudian bisa mengevaluasi fungsi loss di mana persamaan diferensial dikodekan, dan menyetel ulang sudut-sudut dalam loop hybrid, seperti yang ditunjukkan berikut ini. Solusi percobaan secara bertahap semakin mendekati solusi sebenarnya hingga kamu mendapatkan hasil yang memuaskan.

![Alur kerja fungsi QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Selain loop hybrid ini, kamu juga bisa menggabungkan beberapa optimizer secara berantai. Ini berguna ketika kamu ingin optimizer global menemukan sekumpulan sudut yang bagus, lalu optimizer yang lebih halus mengikuti gradien menuju sekumpulan sudut tetangga terbaik. Dalam kasus computational fluid dynamics (CFD), urutan optimasi default menghasilkan hasil terbaik — tetapi dalam kasus material deformation (MD), meskipun default memberikan hasil yang baik, kamu bisa mengonfigurasinya lebih lanjut untuk manfaat spesifik masalah.

Perhatikan bahwa untuk setiap variabel fungsi, kita menentukan jumlah qubit (yang bisa kamu ubah-ubah). Dengan menumpuk 10 sirkuit identik dan mengevaluasi 10 observabel identik pada qubit berbeda sepanjang satu sirkuit besar, kamu bisa melakukan mitigasi noise dalam proses optimasi CMA, mengandalkan metode noise learner, dan secara signifikan mengurangi jumlah shot yang dibutuhkan.
## Input/output
### Computational Fluid Dynamics
Persamaan Burgers' tanpa viskositas memodelkan aliran fluida non-viskos sebagai berikut:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ merepresentasikan medan kecepatan fluida. Kasus penggunaan ini memiliki kondisi batas temporal: kamu bisa memilih kondisi awal lalu membiarkan sistem berelaksasi. Saat ini, satu-satunya kondisi awal yang diterima adalah fungsi linear: $ax + b$.

Argumen untuk persamaan diferensial CFD berada pada grid tetap, sebagai berikut:

- $t$ berada antara 0 dan 0.95 dengan 30 titik sampel. $x$ berada antara 0 dan 0.95 dengan ukuran langkah 0.2375.

### Deformasi Material
Kasus penggunaan ini berfokus pada deformasi hypoelastic dengan uji tarik satu dimensi, di mana sebuah batang yang terpasang di ruang ditarik di ujung lainnya. Kami mendeskripsikan masalahnya sebagai berikut:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ merepresentasikan modulus bulk material yang direntangkan, $n$ eksponen dari hukum pangkat, $b$ gaya per satuan massa, $\epsilon_0$ batas tegangan proporsional, $\sigma_0$ batas regangan proporsional, $u$ fungsi tegangan, dan $\sigma$ fungsi regangan.

Batang yang dipertimbangkan memiliki panjang satuan. Kasus penggunaan ini memiliki kondisi batas untuk tegangan permukaan $t$, atau jumlah kerja yang dibutuhkan untuk meregangkan batang.

Argumen untuk persamaan diferensial MD berada pada grid tetap, sebagai berikut:

- $x$ berada antara 0 dan 1 dengan ukuran langkah 0.04.
### Input
Untuk menjalankan Qiskit Function QUICK-PDE, kamu bisa menyesuaikan parameter berikut:

| Nama              | Tipe                                                 | Deskripsi                                                                                                                                                                                                                                                                       | Spesifik kasus penggunaan | Contoh                   |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | Toggle untuk memilih sistem persamaan diferensial yang akan diselesaikan                                                                                                                                                                                                        | Tidak                     | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | Parameter persamaan diferensial (lihat tabel berikutnya untuk detail lebih lanjut)                                                                                                                                                                                              | Tidak                     | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | Jumlah qubit per fungsi dan per variabel. Nilai yang dioptimalkan dipilih oleh fungsi, tetapi jika kamu ingin mencoba kombinasi yang lebih baik, kamu bisa mengganti nilai default                                                                                                | Tidak                     | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | Kedalaman ansatz per fungsi. Nilai yang dioptimalkan dipilih oleh fungsi, tetapi jika kamu ingin mencoba kombinasi yang lebih baik, kamu bisa mengganti nilai default                                                                                                            | Tidak                     | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | Optimizer yang akan digunakan, baik "CMAES" dari [`cma` python library](https://github.com/CMA-ES/pycma) atau salah satu dari [optimizer scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                              | MD                        | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | Jumlah shot yang digunakan untuk menjalankan setiap Circuit. Karena beberapa langkah optimasi diperlukan, panjang daftar harus sama dengan jumlah optimizer yang digunakan (4 untuk CFD). Default ke `[50_000] * nb_optimizers` untuk MD dan `[5_00, 2_000, 5_000, 10_000]` untuk CFD | Tidak                     | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | Opsi yang akan diteruskan ke optimizer. Detail input ini bergantung pada optimizer yang digunakan; untuk spesifik, lihat dokumentasi optimizer yang digunakan                                                                                                                    | MD                        | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | Apakah akan memulai dengan sudut acak atau sudut yang dipilih secara cerdas. Perhatikan bahwa sudut yang dipilih secara cerdas mungkin tidak berfungsi dalam 100% kasus. Default ke `"PHYSICALLY_INFORMED"`.                                                                     | Tidak                     | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | Nama Backend yang akan digunakan.                                                                                                                                                                                                                                               | Tidak                     | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | Mode eksekusi yang akan digunakan. Default ke `"job"`.                                                                                                                                                                                                                          | Tidak                     | `"job"`                  |

Parameter persamaan diferensial (parameter fisik dan kondisi batas) harus mengikuti format berikut:

| Kasus penggunaan | Kunci       | Tipe nilai | Deskripsi                                      | Contoh |
| ---------------- | ----------- | ---------- | ---------------------------------------------- | ------- |
| CFD              | `a`         | `float`    | Koefisien nilai awal $u$                        | `1.0`   |
| CFD              | `b`         | `float`    | Offset nilai awal $u$                           | `1.0`   |
| MD               | `t`         | `float`    | tegangan permukaan                              | `12.0`  |
| MD               | `K`         | `float`    | modulus bulk                                    | `100.0` |
| MD               | `n`         | `int`      | hukum pangkat                                   | `4.0`   |
| MD               | `b`         | `float`    | gaya per satuan massa                           | `10.0`  |
| MD               | `epsilon_0` | `float`    | batas tegangan proporsional                     | `0.1`   |
| MD               | `sigma_0`   | `float`    | batas regangan proporsional                     | `5.0`   |
### Output
Outputnya adalah dictionary berisi daftar titik sampel, serta nilai-nilai fungsi di setiap titik tersebut:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

Bentuk array solusi bergantung pada sampel variabel:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

Pemetaan antara titik sampel variabel fungsi dan dimensi array solusi dilakukan dalam urutan alfanumerik nama variabel. Misalnya, jika variabelnya adalah `"t"` dan `"x"`, sebuah baris dari `solution["functions"]["u"]` merepresentasikan nilai solusi untuk `"t"` yang tetap, dan sebuah kolom dari `solution["functions"]["u"]` merepresentasikan nilai solusi untuk `"x"` yang tetap.

Berikut adalah contoh cara mendapatkan nilai fungsi untuk sekumpulan koordinat tertentu:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Benchmark

Tabel berikut menyajikan statistik dari berbagai run fungsi kami.

| Contoh                        | Jumlah qubit | Inisialisasi          | Error     | Total waktu (menit) | Penggunaan runtime (menit) |
| ----------------------------- | ------------ | --------------------- | --------- | ------------------- | -------------------------- |
| Persamaan Burgers' tanpa viskositas | 50      | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                  | 25                         |
| Uji tarik 1D Hypoelastic      | 18           | `RANDOM`              | $10^{-2}$ | 123                 | 100                        |

## Mulai

Isi [formulir untuk meminta akses ke fungsi QUICK-PDE.](https://forms.cloud.microsoft/e/3Wi9cbjQPK) Kemudian, dengan asumsi kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokal, pilih fungsinya sebagai berikut:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Contoh

Untuk memulai, coba salah satu contoh berikut:

### Computational Fluid Dynamics (CFD)

Ketika kondisi awal ditetapkan ke $u(0,x) = x$, hasilnya adalah sebagai berikut:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Periksa [status](/guides/functions#check-job-status) workload Qiskit Function kamu atau ambil [hasil](/guides/functions#retrieve-results) sebagai berikut:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Output dari sel kode sebelumnya](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Deformasi Material
Kasus penggunaan deformasi material memerlukan parameter fisik material dan gaya yang diterapkan, sebagai berikut:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![Output dari sel kode sebelumnya](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## Ambil pesan error

Jika status workload kamu adalah `ERROR`, gunakan `job.error_message()` untuk mengambil pesan error guna membantu debugging, sebagai berikut: